# Ecuación de ondas 1D: solución


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def u0(x):
    return np.sin(np.pi * x)


def v0(x):
    return np.zeros_like(x)


def exact_solution(x, t, c=1.0):
    return np.cos(np.pi * c * t) * np.sin(np.pi * x)


def build_grid(c=1.0, L=1.0, T=0.6, N=220, sigma_obj=0.9):
    dx = L / N
    dt = sigma_obj * dx / c
    nt = int(np.ceil(T / dt))
    dt = T / nt
    sigma = c * dt / dx
    x = np.linspace(0.0, L, N + 1)
    return x, dx, dt, nt, sigma


def first_step(u_prev, v_init, sigma, dt):
    u_curr = u_prev.copy()
    u_curr[1:-1] = (
        u_prev[1:-1]
        + dt * v_init[1:-1]
        + 0.5 * sigma**2 * (u_prev[2:] - 2.0 * u_prev[1:-1] + u_prev[:-2])
    )
    u_curr[0] = 0.0
    u_curr[-1] = 0.0
    return u_curr


def wave_step(u_curr, u_prev, sigma):
    u_next = u_curr.copy()
    u_next[1:-1] = 2.0 * u_curr[1:-1] - u_prev[1:-1] + sigma**2 * (
        u_curr[2:] - 2.0 * u_curr[1:-1] + u_curr[:-2]
    )
    u_next[0] = 0.0
    u_next[-1] = 0.0
    return u_next


def solve_wave(c=1.0, L=1.0, T=0.6, N=220, sigma_obj=0.9, store_every=5):
    x, dx, dt, nt, sigma = build_grid(c=c, L=L, T=T, N=N, sigma_obj=sigma_obj)
    u_prev = u0(x)
    v_init = v0(x)
    u_curr = first_step(u_prev, v_init, sigma, dt)

    t_hist = [0.0, dt]
    U_hist = [u_prev.copy(), u_curr.copy()]

    for n in range(1, nt):
        u_next = wave_step(u_curr, u_prev, sigma)
        u_prev, u_curr = u_curr, u_next
        t_next = (n + 1) * dt
        if (n + 1) % store_every == 0 or (n + 1) == nt:
            t_hist.append(t_next)
            U_hist.append(u_curr.copy())

    return x, u_curr, nt * dt, sigma, np.array(t_hist), np.array(U_hist)


c = 1.0
x, u_num, t_final, sigma, t_hist, U_hist = solve_wave(c=c)
u_ex = exact_solution(x, t_final, c=c)
err_l2 = np.sqrt(np.mean((u_num - u_ex) ** 2))
print(f"sigma={sigma:.6f}")
print(f"t_final={t_final:.6f}")
print(f"error L2={err_l2:.3e}")


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x, u_num, lw=2.2, label="Numérica")
plt.plot(x, u_ex, "--", lw=2.0, label="Exacta")
plt.title("Ecuación de ondas 1D en t_final")
plt.xlabel("x")
plt.ylabel("u(x,t_final)")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


## Perfiles de la cuerda en distintos tiempos


In [ ]:
plt.figure(figsize=(8, 4))
for t_ref in [0.00, 0.15, 0.30, 0.45, 0.60]:
    idx = int(np.argmin(np.abs(t_hist - t_ref)))
    plt.plot(x, U_hist[idx], lw=1.8, label=f"t={t_hist[idx]:.3f}")
plt.title("Ondas 1D: evolución de perfiles")
plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


## Efecto de la condición CFL


In [ ]:
for sig in [0.7, 0.9, 1.05]:
    x_s, u_s, t_s, sigma_s, _, _ = solve_wave(c=c, L=L, T=0.6, N=N, sigma_obj=sig, store_every=20)
    print(f"sigma objetivo={sig:.2f} -> sigma efectivo={sigma_s:.4f}, max|u|={np.max(np.abs(u_s)):.3f}")
